<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/notes/8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №8: Вейвлет-преобразования

## 1. Библиотека PyWavelets

```python
!pip install -q PyWavelets      # в Colab обычно уже установлена
import pywt
pywt.wavelist(kind='continuous')  # вейвлеты для CWT
pywt.wavelist(kind='discrete')    # вейвлеты для DWT
```

**Имена вейвлетов:**
- `'haar'`, `'dbN'` (Добеши, N нулевых моментов, фильтр длины 2N), `'symN'` (симлеты), `'coifN'` (койфлеты), `'biorNr.Nd'` (биортогональные) – для DWT;
- `'mexh'` (мексиканская шляпа), `'morl'` (вещественный Морле), `'gausN'` (производные гауссиана), `'cmorB-C'` (комплексный Морле с шириной B и центральной частотой C) – для CWT.

## 2. Непрерывное преобразование (CWT)

```python
fs = 1000
freqs = np.geomspace(5, 400, 150)                      # нужные частоты, Гц
scales = pywt.central_frequency('cmor1.5-1.0') * fs / freqs
W, f = pywt.cwt(x, scales, 'cmor1.5-1.0', sampling_period=1/fs, method='fft')
plt.pcolormesh(t, f, np.abs(W), shading='auto')
plt.yscale('log')
```

- Связь масштаба и частоты: $f = \dfrac{f_c}{a\,\Delta t}$, где $f_c$ – центральная частота вейвлета (`pywt.central_frequency`), $\Delta t = 1/f_s$.
- `W` имеет форму `(число масштабов, длина сигнала)`; для комплексного вейвлета значения комплексные, для анализа берут `np.abs(W)`.
- `method='fft'` значительно быстрее для длинных сигналов и большого числа масштабов.
- **Конус влияния** для `cmorB-C`: на масштабе $a$ краевые эффекты распространяются на $a\sqrt{B/2}$ отсчётов от краёв записи.
- Ширина $B$: больше $B$ – больше осцилляций в вейвлете, лучше разрешение по частоте и хуже по времени.

## 3. Дискретное преобразование (DWT)

```python
cA, cD = pywt.dwt(x, 'db4', mode='symmetric')       # один уровень
coeffs = pywt.wavedec(x, 'db4', level=5)             # [cA5, cD5, cD4, cD3, cD2, cD1]
x_rec = pywt.waverec(coeffs, 'db4')                  # может оказаться на 1 отсчёт длиннее x
J = pywt.dwt_max_level(len(x), 'db4')                # максимальное число уровней
```

- **Порядок коэффициентов** в `wavedec`: сначала аппроксимация самого грубого уровня, затем детали от грубых к мелким. Деталь $d_j$ (первый уровень – самый мелкий) соответствует полосе $[f_s/2^{j+1},\, f_s/2^j]$.
- **Режим продолжения** `mode`: `'symmetric'` (по умолчанию), `'periodization'` (длина коэффициентов ровно вдвое меньше на каждом уровне – удобно для сжатия), `'zero'`, `'smooth'` и др.
- **Фильтры:** `w = pywt.Wavelet('db4')`; `w.dec_lo`, `w.dec_hi` – анализ, `w.rec_lo`, `w.rec_hi` – синтез; `w.wavefun(level=8)` – значения $\varphi$ и $\psi$.
- Ручная реализация одного уровня (совпадает с `mode='zero'`):
```python
a = np.convolve(x, w.dec_lo)[1::2]
d = np.convolve(x, w.dec_hi)[1::2]
```

## 4. Вклад отдельного уровня

```python
def component(coeffs, i, wavelet):
    c = [np.zeros_like(ci) for ci in coeffs]
    c[i] = coeffs[i]
    return pywt.waverec(c, wavelet)
```

## 5. Пороговая обработка

```python
sigma = np.median(np.abs(coeffs[-1])) / 0.6745       # оценка шума по деталям 1-го уровня
lam = sigma * np.sqrt(2 * np.log(len(x)))            # универсальный порог
coeffs_t = [coeffs[0]] + [pywt.threshold(d, lam, mode='soft') for d in coeffs[1:]]
```

- Жёсткий порог (`'hard'`) обнуляет малые коэффициенты и не меняет большие; мягкий (`'soft'`) дополнительно уменьшает большие на $\lambda$ – результат глаже, но амплитуды занижены.
- Аппроксимацию обычно не обрабатывают.

## 6. Сжатие: работа со всеми коэффициентами сразу

```python
arr, slices = pywt.coeffs_to_array(coeffs)           # все коэффициенты в одном массиве
k = int(p * arr.size)
thr = np.sort(np.abs(arr).ravel())[-k]
arr_c = np.where(np.abs(arr) >= thr, arr, 0)
coeffs_c = pywt.array_to_coeffs(arr_c, slices, output_format='wavedec')   # 'wavedec2' для 2D
```

Для изображений: `pywt.wavedec2(img, 'bior4.4', level=3)`, `pywt.waverec2`. Каждый уровень деталей – кортеж из трёх субполос (горизонтальные, вертикальные, диагональные детали). Аппроксимацию `arr[slices[0]]` при сжатии лучше сохранять целиком.

**PSNR** для 8-битного изображения: `10*np.log10(255**2 / np.mean((img_rec - img)**2))`.

## 7. Нулевые моменты

Вейвлет с $p$ нулевыми моментами даёт нулевые детали для многочленов степени меньше $p$. У Добеши dbN $p = N$: `pywt.Wavelet('db3').vanishing_moments_psi`.

## 8. Типичные ошибки

1. **Перепутан порядок уровней:** `coeffs[-1]` – самые мелкие детали $d_1$, а не самые крупные.
2. **Длина после `waverec`:** при нечётной длине или режиме, отличном от `'periodization'`, восстановленный сигнал может быть длиннее исходного – обрежьте `x_rec[:len(x)]`.
3. **Масштабы вместо частот:** `pywt.cwt` принимает масштабы; частоты нужно пересчитать через центральную частоту вейвлета.
4. **Линейная ось частот на скалограмме:** при геометрической сетке масштабов используйте логарифмическую ось.
5. **Вещественный вейвлет для частотного анализа:** у `'morl'` и `'mexh'` модуль коэффициентов осциллирует во времени; для оценки огибающей используйте `'cmor'`.
6. **Число уровней:** уровней больше `pywt.dwt_max_level` вызывают предупреждение и сильные краевые эффекты.